# 5. Modelo propio: algoritmo, propiedad y explicación

## Idea y alcance

Se busca una función aditiva que permita inspeccionar términos univariados y pocas interacciones. El ranking intenta favorecer ordenamiento positivo–negativo; L2 limita coeficientes. La combinación no garantiza calibración, estabilidad de las interacciones ni mejora predictiva.

![Esquema del algoritmo propio](_static/entregable3/flujo_modelo.svg)

## Fundamento del modelo propio: alcance verificable

La tarea es clasificar caries registrada: $y_i\in\{0,1\}$ y $x_i$ contiene los 23 predictores conservados. No se pronostica incidencia futura. Para una representación aprendida solo con el ajuste, $z_i=\phi(x_i)$, el score es $s_i=b+z_i^\top w$ y la salida inicial es $\sigma(s_i)=1/(1+e^{-s_i})$.

Las numéricas usan splines cúbicos; las categóricas, one-hot. Se consideran hasta cinco productos entre numéricas estandarizadas. La selección por asociación con residuos es una heurística interna: no acredita control de falsos descubrimientos ni estabilidad inferencial.

Para $M$ pares positivos-negativos fijos, la función implementada es:

$$
L(b,w)=(1-\alpha)\frac1n\sum_i\left[\log(1+e^{s_i})-y_i s_i\right]+\frac{\alpha}{M}\sum_{(p,q)}\log(1+e^{-(s_p-s_q)})+\frac{\lambda}{2}\lVert w\rVert_2^2.
$$

La BCE favorece el ajuste binario; el ranking favorece scores mayores en positivos; L2 penaliza los coeficientes, no el intercepto. Las pérdidas se normalizan por separado. La mezcla no garantiza probabilidades calibradas.

Para un par, $d=s_p-s_q$ y $r=\sigma(-d)$: la derivada respecto a $s_p$ es $-r$ y respecto a $s_q$, $+r$. La parte BCE aporta $(\sigma(s_i)-y_i)/n$. El gradiente de $w$ combina $Z^\top$ por las derivadas del score y $\lambda w$. Se entrega a L-BFGS-B; el máximo de iteraciones y el indicador de convergencia se registran.

**Propiedad condicional.** Con representación y pares fijos, $0\leq\alpha<1$, $\lambda>0$ y ambas clases presentes, la pérdida es estrictamente convexa en $(b,w)$. Cada término logístico tiene segunda derivada no negativa; L2 es positiva en cualquier dirección con componente de coeficientes no nula. En la dirección exclusiva del intercepto, la BCE aporta curvatura estrictamente positiva para scores finitos. Ambas clases y L2 impiden escapar al infinito sin aumentar el objetivo, por lo que existe un mínimo único. Esto no convierte en convexa la selección previa de interacciones ni garantiza que el criterio numérico haya alcanzado el óptimo.

**Complejidad.** Con $d$ columnas transformadas, $p$ numéricas, $B$ submuestras, $I_0$ iteraciones preliminares e $I$ finales, el trabajo principal es del orden de $I_0nd+Bn p^2+I(nd+M)$. La matriz de todos los productos introduce memoria del orden de $np^2$; las bases, $nd$. La inferencia de $m$ filas escala con su transformación y $md$. Son órdenes del algoritmo implementado, no una medición de memoria del propio ni un estudio empírico de escalamiento.

**Pseudocódigo.** (1) Validar esquema y dos clases. (2) Aprender imputación, splines, codificación y escalas solo en ajuste. (3) Fijar pares con semilla. (4) Ajustar aditivo preliminar y seleccionar interacciones dentro de ajuste. (5) Minimizar la pérdida con gradiente y registrar convergencia. (6) Transformar nuevas filas con los objetos guardados y calcular score/sigmoide; si se calibra, usar calibrador aprendido con OOF interno.

**Resultado y originalidad.** AUC medio V1 propio: 0,605062; aditivo sin ambos componentes: 0,606151. Las ablaciones no prueban ventaja de añadir ranking/interacciones. Implementación propia y una propiedad matemática verificable no demuestran novedad científica (R1) ni superioridad (R3). La revisión sistemática de novedad sigue pendiente.

**Pruebas nuevas.** Cinco pruebas pasaron: aprendizaje de regla sintética en filas no usadas al ajustar, repetibilidad con igual semilla, clone sin estado, rechazo de entradas excluidas/clase única y gradiente por diferencias finitas. No sustituyen validación clínica ni reproducen todos los entrenamientos históricos.


## Formulación numerada y gradiente

Para $z_i=\phi(x_i)$ y parámetros $\theta=(b,w)$:

$$
s_i=b+z_i^\top w,\qquad p_i=\sigma(s_i)=\frac{1}{1+\exp(-s_i)}. \tag{5.1}
$$

Con $\mathcal P$ fijo de $M$ pares $(p,q)$, positivos y negativos:

$$
L(\theta)=\frac{1-\alpha}{n}\sum_{i=1}^n[\log(1+e^{s_i})-y_i s_i]+\frac{\alpha}{M}\sum_{(p,q)\in\mathcal P}\log(1+e^{-(s_p-s_q)})+\frac{\lambda}{2}\Vert w\Vert_2^2. \tag{5.2}
$$

Si $r_{pq}=\sigma(-(s_p-s_q))/M$, el ranking aporta $-r_{pq}$ al score positivo y $+r_{pq}$ al negativo. Agrupar esos aportes por índice produce un vector $g_s$, sin construir diferencias de características para cada par. El gradiente es:

$$
\nabla_w L=Z^\top g_s+\lambda w,\qquad
\partial_b L=\frac{1-\alpha}{n}\sum_i(\sigma(s_i)-y_i). \tag{5.3}
$$

El ranking cancela el intercepto porque solo depende de diferencias de scores. En la implementación, `logaddexp` evita exponentes inestables y `bincount` acumula gradientes por fila.

## Pseudocódigo formal

**Entrada:** $X,y$, $0\le\alpha<1$, $\lambda>0$, número de nudos, máximo de interacciones, número de pares, semilla, tolerancia y límite de iteraciones. **Salida:** transformaciones ajustadas, pares de variables seleccionados, coeficientes, intercepto y diagnóstico de convergencia.

1. Validar esquema, columnas excluidas, tamaños y presencia de ambas clases.
2. Aprender imputación, bases spline, one-hot y escalas solo en el ajuste actual.
3. Muestrear con reemplazo $M$ pares positivo–negativo y mantenerlos fijos.
4. Ajustar el aditivo preliminar y calcular residuos internos.
5. Explorar productos numéricos mediante submuestras de perfiles del ajuste; ordenar por frecuencia/asociación y conservar hasta cinco.
6. Optimizar (5.2) con gradiente (5.3), L-BFGS-B y máximo de iteraciones.
7. Registrar éxito, iteraciones, pérdida y gradiente. Un fallo de convergencia detiene la evaluación.
8. Para nuevas filas, aplicar las transformaciones guardadas y calcular (5.1).

La parada es la condición numérica de L-BFGS-B o el límite de iteraciones; no una prueba de solución exacta. La representación y los pares fijos permiten la propiedad convexa demostrada arriba, pero la selección de interacciones queda fuera de ella.

## Pruebas y ablaciones

Siete pruebas pasaron: regla sintética conocida, repetibilidad, clone sin estado, entradas inválidas/clase única, gradiente por diferencias finitas, AUC ponderada con empates frente a sklearn y jackknife exacto frente a retirar cada perfil. No son una certificación clínica.

## 3. Contribución implementada y ablaciones

Desarrollé un modelo que combina funciones spline, interacciones numéricas limitadas,
pérdida logística y ranking por pares, con regularización L2.
La implementación y sus gradientes fueron verificados. La novedad científica
de la combinación no está establecida.

La selección de interacciones utiliza residuos de ajuste. Las repeticiones
submuestrean perfiles pero reutilizan el modelo preliminar y sus residuos:
no equivalen a reentrenar todo el procedimiento de selección en cada repetición.

Las ablaciones conservadas permiten evaluar variantes sin interacciones,
sin ranking y aditiva. Sus espacios de regularización no son idénticos;
comparan procedimientos ajustados y no aíslan perfectamente cada componente.

### Resultados V1 conservados

<table class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th>Modelo</th>
      <th>Folds</th>
      <th>AUC medio</th>
      <th>DE entre folds</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Aditivo</td>
      <td>5</td>
      <td>0.606151</td>
      <td>0.009029</td>
    </tr>
    <tr>
      <td>Propio</td>
      <td>5</td>
      <td>0.605062</td>
      <td>0.007275</td>
    </tr>
    <tr>
      <td>Sin_interacciones</td>
      <td>5</td>
      <td>0.605092</td>
      <td>0.006740</td>
    </tr>
    <tr>
      <td>Sin_ranking</td>
      <td>5</td>
      <td>0.605964</td>
      <td>0.009598</td>
    </tr>
  </tbody>
</table>

Las ablaciones V1 se mantienen: el aditivo sin ranking ni interacciones alcanzó AUC medio aproximadamente 0,606151, frente a 0,605062 del propio completo. No prueban que añadir ambos componentes ayude. Los hiperparámetros V1 se optimizaron por grilla dentro de la CV anidada; el suplemento pareado conserva cuatro candidatos propios y cuatro por referencia.

## Interpretación propia y mejor referencia final

![Importancia aditiva descriptiva de dos modelos finales](_static/entregable3/explicacion_global_finales.png)

Se suman las contribuciones de bases por variable, manteniendo las interacciones aparte. Son 500 perfiles de desarrollo y objetos ya reajustados; no observaciones externas. Las escalas de margen de modelos distintos no permiten comparar magnitudes de importancia absoluta entre paneles. La interpretación es predictiva, no causal.

![Explicación exacta de un perfil sintético](_static/entregable3/explicacion_local_propio.png)

Valor base más todas las contribuciones reconstruye el margen con error inferior a $10^{-12}$. La sigmoide original no se presenta como riesgo clínico calibrado. Esta descomposición es una alternativa justificada a SHAP para una función aditiva explícita; no es una estimación de efectos de intervenir sobre una persona.

[Método de explicación](_static/entregable3/explicacion_metodo.json) · [Contribuciones del caso](_static/entregable3/explicacion_local_propio.csv)

## Código del núcleo

La copia siguiente coincide con `proyecto_final_caries/modelo.py`. `objetivo.py` contiene la pérdida y su gradiente. No se oculta el uso de transformadores y optimización SciPy; la lógica propia está identificada, y el autor debe demostrar comprensión y capacidad de modificarla durante la defensa.

````{dropdown} Ver la implementación de la pérdida y el gradiente
```{literalinclude} src/objetivo.py
:language: python
:linenos:
```
````
